# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [1]:
# !pip install langchain_community langchain_experimental langchain_openai

In [2]:
from typing import Optional

from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [3]:
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_classic.agents import AgentType

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

# OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [5]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5"
)

C:\Users\felip\AppData\Local\Temp\ipykernel_30936\93859234.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(


In [15]:
# Define your embedding model
# embeddings_model = OpenAIEmbeddings()
# Initialize the vectorstore as empty
import faiss

# embedding_size = 1536
# index = faiss.IndexFlatL2(embedding_size)
# vectorstore = FAISS(embeddings_model.embed_query, index, InMemoryDocstore({}), {})

# Dynamically detect embedding dimension
embedding_size = len(embeddings_model.embed_query("hello"))

# Create FAISS index
index = faiss.IndexFlatL2(embedding_size)

# Create vectorstore
vectorstore = FAISS(
    embedding_function=embeddings_model,
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [16]:
from langchain_community.llms import Ollama

llm = Ollama(model="mistral",temperature=0)

In [17]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain_classic.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
# todo_chain = LLMChain(llm=OpenAI(temperature=0), prompt=todo_prompt)
todo_chain = LLMChain(llm=llm, prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

In [18]:
# llm = OpenAI(temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [19]:
OBJECTIVE = "Write a weather report for SF today"

In [22]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    # max_iterations=max_iterations,
    max_iterations=1,
)

In [21]:
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
 Thought: The objective is to create a todo list.
Action: TODO
Action Input: Create a weather report for SF today
Observation:  Title: Weather Report for San Francisco (Today)

1. Gather Data Sources:
   - Identify reliable weather data sources such as National Weather Service, Weather.com, OpenWeatherMap, etc.

2. Research Current Conditions:
   - Check the current temperature, humidity, wind speed and direction, precipitation, and air quality index for San Francisco.

3. Forecast Future Conditions:
   - Obtain forecasts for the rest of the day from multiple sources to ensure accuracy.
   - Include information about expected changes in weather conditions throughout the day (e.g., temperature highs and lows, precipitation chances).

4. Analyze Weather Patterns:
   - Identify any unusual or significant weather patterns that may affect San Francisco today.
   - Cons

KeyboardInterrupt: 

In [23]:
OBJECTIVE = "Who is the strongest Belmont vampire hunter?"
baby_agi({"objective": OBJECTIVE})


*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
 Question: Make a todo list for determining who is the strongest Belmont vampire hunter from the Castlevania series.

Thought: To determine who is the strongest Belmont vampire hunter, I need to gather information about each character's abilities, strengths, and achievements in the Castlevania series. Then, I will compare them to find out who is the strongest. First, let's create a todo list for this task.

Action: TODO

Action Input:
- Research the background and abilities of Simon Belmont (Castlevania, 1986)
- Research the background and abilities of Trevor Belmont (Castlevania III: Dracula's Curse, 1989)
- Research the background and abilities of Alucard (Castlevania II: Simon's Quest, 1987)
- Research the background and abilities of Richter Belmont (Castlevania: Rondo of Blood, 1993)
- Research the background and abilities of Soleiyu Belmont (Castlevania Chron

ValueError: Tried to add ids that already exist: {'result_1_0'}